# ARC-AGI-2 — Test-Time Bayesian Program Induction (offline submission)

This notebook produces a `submission.json` for the ARC Prize 2026 ARC-AGI-2
competition using the **offline** solver from the `arc_solver` package:
fitted analytic proposals + MDL-ordered enumerative program search, with a
pass@2-aware posterior-mode selector. It runs **fully offline** (no internet,
no API calls) inside the Kaggle sandbox.

**Method in one line:** for each task we induce a short program (over a compact
grid DSL) that reproduces *all* demonstration pairs, weight admissible programs
by a Minimum-Description-Length prior, and emit the posterior-mode output as
`attempt_1` and the most probable *disagreeing* output as `attempt_2`.

**Kaggle input (exact path this notebook expects at rerun):**
`/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json` — a single
combined JSON file `{"<task_id>": {"train": [...], "test": [...]}}` covering
all 240 unseen test tasks (test pairs have no `"output"`; it is hidden). This
notebook does **not** glob a directory of per-task files — the competition
does not mount the data that way.

**Internet must be turned OFF** for this notebook (Settings → Internet → Off).
The competition sandbox has no network access at scoring time regardless, but
turning it off locally too is what lets you verify the notebook will actually
run under the same constraints it will be rerun under.

To run on Kaggle, attach this repository's `src/` directory as a dataset
(e.g. `arc-solver-src`) so `arc_solver` is importable — see `DEPLOYMENT.md` for
the exact steps (Kaggle Dataset attach, or inlining the package). The cell
below searches common locations automatically.


In [ ]:
import os, sys

# Make the arc_solver package importable. Attach this repo's code as a Kaggle
# Dataset named "arc-solver-src" (see DEPLOYMENT.md). Kaggle may mount it either
# with a src/ wrapper (/kaggle/input/arc-solver-src/src/arc_solver) or with the
# package at the dataset root (/kaggle/input/arc-solver-src/arc_solver), e.g. if
# the src/ folder's contents were uploaded directly. We search both layouts, so
# the notebook works regardless of how the dataset was created.
CANDIDATES = [
    '/kaggle/input/arc-solver-src/src',
    '/kaggle/input/arc-solver-src',
    '/kaggle/input/arc-solver/src',
    '/kaggle/input/arc-solver',
    os.path.join(os.getcwd(), 'src'),
    os.path.join(os.path.dirname(os.getcwd()), 'src'),
    os.getcwd(),
]
for c in CANDIDATES:
    if os.path.isdir(os.path.join(c, 'arc_solver')):
        sys.path.insert(0, c)
        print('arc_solver from', c)
        break
else:
    raise RuntimeError(
        'arc_solver not found; attach the arc-solver-src dataset (see DEPLOYMENT.md). '
        'Searched: ' + ', '.join(CANDIDATES)
    )

from arc_solver.cli.run_eval import build_engine
from arc_solver.io.loader import load_challenges
from arc_solver.io.submission import SubmissionWriter

In [ ]:
# Locate the combined-challenges JSON. The real competition rerun mounts the
# hidden test set at the first path below. We fall back, in order, to the
# evaluation-challenges file (a dry run against known-format data with the
# same shape) and finally to a locally-built combined file for offline
# development (see DEPLOYMENT.md for how to build one from data/).
CHALLENGES_CANDIDATES = [
    '/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json',
    '/kaggle/input/arc-prize-2026-arc-agi-2/arc-agi_evaluation-challenges.json',
    'data/kaggle_dry_run_challenges.json',
    '../data/kaggle_dry_run_challenges.json',
]
CHALLENGES_PATH = next((p for p in CHALLENGES_CANDIDATES if os.path.isfile(p)), None)
if CHALLENGES_PATH is None:
    raise FileNotFoundError(
        'no combined challenges JSON found; expected the competition to mount '
        'arc-agi_test_challenges.json, or a locally-built dry-run file — see DEPLOYMENT.md'
    )

tasks = load_challenges(CHALLENGES_PATH)
print('challenges file:', CHALLENGES_PATH, '| tasks:', len(tasks))


In [ ]:
import time

# Offline engine: fitted proposals first, then bounded enumerative search.
# Fully offline — numpy only, no network, no LLM calls.
engine = build_engine(max_seconds=10.0, max_programs=20000, max_depth=4)
writer = SubmissionWriter()

TOTAL_BUDGET_S = 11 * 3600   # wall-clock guard, well under the 12h Kaggle limit
start = time.monotonic()
solved = 0

for i, task in enumerate(tasks):
    if time.monotonic() - start > TOTAL_BUDGET_S:
        # Out of time: emit a safe identity/mirror fallback so no task is missing.
        attempts = []
        for p in task.test:
            g = p.input
            mirror = tuple(tuple(reversed(r)) for r in g)
            attempts.append((g, mirror if mirror != g else g))
        writer.add(task.task_id, attempts)
        continue
    result = engine.solve(task)
    solved += int(result.solved_train)
    writer.add(task.task_id, list(result.attempts))
    if (i + 1) % 20 == 0:
        print(f'{i + 1}/{len(tasks)} tasks, {solved} with a program, '
              f'{time.monotonic() - start:.0f}s')

# Write to the exact path Kaggle scores; fall back to a local path off-Kaggle.
OUT_PATH = '/kaggle/working/submission.json' if os.path.isdir('/kaggle/working') else 'submission.json'
writer.write(OUT_PATH)
assert len(writer) == len(tasks), 'every task must be present in the submission'
print('wrote', OUT_PATH, ':', len(writer), 'tasks,', solved, 'with an admissible program')


The full method, theory, and ablations are described in the accompanying
write-up (`WRITEUP.md`). The offline path above is the reproducible core; an
optional LLM-amortised proposer (`arc_solver.search.llm_proposer`) raises
accuracy when an API key is available, but **cannot run inside the Kaggle
no-internet sandbox** — it is disabled here. See `SECURITY.md` and
`docs/RUNBOOK.md` §4 for the gated rollout of that path outside Kaggle.
